# MAS: Multi-Agent System for Airport Anomaly Detection

This notebook implements the Multi-Agent version of the airport anomaly detection system. 
The objective is not only to detect abnormal airport transit patterns, but also to show how a multi-agent architecture can separate the workflow into specialized agents: data extraction, baseline construction, anomaly detection, risk profiling, and reporting by utilizing a **Multi-Agent System (MAS)** built with `LangGraph` and local LLMs via `Ollama`. 

*The agentic uses the same cleaned dataset from classical. Therefore we can have a fair comparison.

### 0.1 Imports and Configuration

Here we import the libraries and define the main settings used across the notebook.  
Keeping paths, random seed, and parameters in one place makes the notebook easier to run and debug.

In [19]:
import pandas as pd
import numpy as np
import operator
import io
import json
import re
import warnings

from pathlib import Path
from typing import TypedDict, Annotated, List

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

warnings.filterwarnings("ignore")


#For Reproducibility
RANDOM_STATE = 42
CONTAMINATION = 0.05
MIN_ROWS_FOR_DETECTION = 10

#Paths
BASE_DIR = Path.cwd()
IO_DIR = BASE_DIR / "io"

#Cleaned datasets
PREPROCESS_DIR = IO_DIR / "preprocessing"
PASSENGER_CLEAN_PATH = PREPROCESS_DIR / "passenger_clean.csv"
CASES_CLEAN_PATH = PREPROCESS_DIR / "cases_clean.csv"

#Raw fallbacks, only used if clean files are missing
RAW_PASSENGER_PATH = IO_DIR / "TIPOLOGIA_VIAGGIATORE.csv"
RAW_CASES_PATH = IO_DIR / "ALLARMI.csv"

AGENT_REPORT_DIR = IO_DIR / "agent_report"
AGENT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

### 0.2 Shared Memory (State)
In an agentic system, the **State** acts as the shared memory. It allows agents to pass structured data (like JSON strings) and natural language messages to one another.

The state keeps the selected airport, the filtered datasets, the baseline table, anomaly results, risk profile, and final report.  
Basically, it is the backpack carried by the agents during the workflow.

In [20]:
class AgentState(TypedDict, total=False):
    messages: Annotated[List[BaseMessage], operator.add]
    perimeter: str
    status: str
    reason: str

    #Data Agent output
    passenger_json: str
    alarm_json: str

    #Baseline Agent output
    baseline_dataframe_json: str

    #Outlier Detection Agent output
    anomaly_results: str
    scored_dataframe_json: str

    #Risk Profiling Agent output
    risk_profile: str

    #Report Agent output
    final_report: str

### 0.3 Utility functions

These helper functions keep the pipeline cleaner.  
They handle date parsing, safe rate calculation, LLM output cleaning, and robust scoring.


In [21]:
#This fuinction is only used when the clean datasets are missing, and is designed to handle the specific date format issues in the raw datasets.
def clean_italian_dates(series):
    month_map = {
        "GEN": "JAN", "FEB": "FEB", "MAR": "MAR", "APR": "APR",
        "MAG": "MAY", "GIU": "JUN", "LUG": "JUL", "AGO": "AUG",
        "SET": "SEP", "OTT": "OCT", "NOV": "NOV", "DIC": "DEC"
    }

    def replace_month(date_str):
        if pd.isna(date_str):
            return date_str
        date_str = str(date_str).upper()
        for it, en in month_map.items():
            date_str = date_str.replace(it, en)
        return date_str

    translated = series.apply(replace_month)
    parsed = pd.to_datetime(translated, format="mixed", dayfirst=True, errors="coerce")

    failed = parsed.isna().sum()
    if failed > 0:
        print(f"Date parsing warning: {failed} values could not be parsed.")

    return parsed

#This function cleans the output from local reasoning models (Removes the tags)
def clean_llm_output(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


#This function prevents division by zero in rate calculations
def safe_rate(numerator, denominator):
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")

    result = pd.Series(np.nan, index=num.index, dtype="float64")

    valid_mask = den.notna() & (den > 0)

    result.loc[valid_mask] = num.loc[valid_mask] / den.loc[valid_mask]

    return result

#This function implements a robust scoring mechanism using percentile clipping, which is less sensitive to outliers than min-max scaling. 
#It is used by the Risk Profiling Agent to score risk levels based on the distribution of the data.
def robust_score(series, lower_q=0.01, upper_q=0.99):
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)

    if s.notna().sum() == 0:
        return pd.Series(0.0, index=s.index)

    lo = s.quantile(lower_q)
    hi = s.quantile(upper_q)

    if pd.isna(lo) or pd.isna(hi) or hi <= lo:
        return pd.Series(0.0, index=s.index)

    return ((s.clip(lo, hi) - lo) / (hi - lo)).fillna(0.0)

#This function standardizes text columns
def standardize_text_column(df, col):
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper().str.strip()
    return df

## 1. Data Agent
The Data Agent is the entry point of the system. It selects the airport perimeter, for example `FCO`.  
It loads the cleaned passenger and alarm data, filters the selected airport, and prepares the data for the next agent.

### 1.1 Data Agent Setup

In [22]:
def fetch_security_context(perimeter: str):
    p_up = perimeter.upper().strip()
    #Load cleaned data
    if PASSENGER_CLEAN_PATH.exists():
        passenger = pd.read_csv(PASSENGER_CLEAN_PATH)
    else:
        raise FileNotFoundError(
            f"Missing {PASSENGER_CLEAN_PATH}. "
            "Run the classical preprocessing cells first."
        )

    if CASES_CLEAN_PATH.exists():
        cases = pd.read_csv(CASES_CLEAN_PATH)
    else:
        raise FileNotFoundError(
            f"Missing {CASES_CLEAN_PATH}. "
            "Run the classical preprocessing cells first."
        )

    #Drop accidental index columns
    passenger = passenger.drop(columns=["Unnamed: 0"], errors="ignore")
    cases = cases.drop(columns=["Unnamed: 0"], errors="ignore")

    #Standardize dates
    passenger["departure_date"] = pd.to_datetime(
        passenger["departure_date"], errors="coerce"
    )
    cases["departure_date"] = pd.to_datetime(
        cases["departure_date"], errors="coerce"
    )

    passenger = passenger.dropna(subset=["departure_date"]).copy()
    cases = cases.dropna(subset=["departure_date"]).copy()

    #Standardize airport/city text
    for col in [
        "arrival_airport_code", "departure_airport_code",
        "arrival_city", "departure_city"
    ]:
        passenger = standardize_text_column(passenger, col)

    for col in [
        "arrival_airport_code", "departure_airport_code",
        "arrival_city_name", "departure_city_name"
    ]:
        cases = standardize_text_column(cases, col)

    # Filter selected perimeter
    p_mask = (
        passenger["arrival_airport_code"].astype(str).str.upper().eq(p_up)
        | passenger["arrival_city"].astype(str).str.upper().str.contains(p_up, na=False)
    )

    c_mask = (
        cases["arrival_airport_code"].astype(str).str.upper().eq(p_up)
        | cases["arrival_city_name"].astype(str).str.upper().str.contains(p_up, na=False)
    )

    p_df = passenger.loc[p_mask].copy()
    c_df = cases.loc[c_mask].copy()

    # Numeric cleaning and data-quality flags
    count_cols = [
        "passengers_entries_count",
        "passengers_investigated_count",
        "passengers_flagged_count"
    ]

    p_df["negative_count_issue"] = False

    for col in count_cols:
        if col in p_df.columns:
            p_df[col] = pd.to_numeric(p_df[col], errors="coerce").fillna(0)
            p_df["negative_count_issue"] = p_df["negative_count_issue"] | (p_df[col] < 0)
            p_df[col] = p_df[col].clip(lower=0)

    if "total_flights" in c_df.columns:
        c_df["total_flights"] = pd.to_numeric(c_df["total_flights"], errors="coerce").fillna(0)
        c_df["total_flights"] = c_df["total_flights"].clip(lower=0)

    #Logical consistency check
    p_df["logical_count_issue"] = (
        (p_df["passengers_flagged_count"] > p_df["passengers_investigated_count"])
        | (p_df["passengers_investigated_count"] > p_df["passengers_entries_count"])
    )

    p_df["data_quality_issue"] = (
        p_df["negative_count_issue"] | p_df["logical_count_issue"]
    )

    #Route features, same idea as classical pipeline
    p_df["date"] = p_df["departure_date"].dt.floor("D")
    c_df["date"] = c_df["departure_date"].dt.floor("D")

    p_df["route_airport"] = (
        p_df["departure_airport_code"].astype(str) + "_" +
        p_df["arrival_airport_code"].astype(str)
    )

    p_df["route_city"] = (
        p_df["departure_city"].astype(str) + "_" +
        p_df["arrival_city"].astype(str)
    )

    p_df["route_country"] = (
        p_df["departure_country_code"].astype(str) + "_" +
        p_df["arrival_country_code"].astype(str)
    )

    c_df["route_airport"] = (
        c_df["departure_airport_code"].astype(str) + "_" +
        c_df["arrival_airport_code"].astype(str)
    )

    c_df["route_city"] = (
        c_df["departure_city_name"].astype(str) + "_" +
        c_df["arrival_city_name"].astype(str)
    )

    c_df["route_country"] = (
        c_df["departure_country_code"].astype(str) + "_" +
        c_df["arrival_country_code"].astype(str)
    )

    print(f"DATA AGENT: {p_up}")
    print(f"Passenger rows selected: {len(p_df)}")
    print(f"Case rows selected: {len(c_df)}")

    return p_df, c_df

### 1.2 Data Agent node
This node receives the user input and extracts the airport code.  
Then it calls the data loading function and stores the filtered datasets in the shared state.

In [23]:
def data_agent_node(state: AgentState):
    """
    Data Agent:
    - extracts the target IATA code
    - fetches and cleans the relevant rows
    - hands structured JSON to the Baseline Agent

    The LLM is kept for MAS structure, but the final extraction is guarded
    so it cannot modify the user's code into another airport.
    """
    llm = ChatOllama(model="llama3.2:3b", temperature=0)

    user_request = state["messages"][-1].content

    prompt = f"""
Extract the airport IATA code ONLY from the text between <USER_REQUEST> and </USER_REQUEST>.

<USER_REQUEST>
{user_request}
</USER_REQUEST>

Rules:
- Return ONLY one 3-letter uppercase IATA code explicitly present in USER_REQUEST.
- Do NOT use examples.
- Do NOT invent, guess, correct, or replace the code.
- If no 3-letter code appears, return NONE.

Answer only the code or NONE.
"""

    raw = llm.invoke([HumanMessage(content=prompt)]).content.strip().upper()

    # deterministic safety: prefer codes from the actual user request, not from the LLM output
    user_matches = re.findall(r"\b[A-Z]{3}\b", user_request.upper())
    llm_matches = re.findall(r"\b[A-Z]{3}\b", raw)

    if user_matches:
        perimeter = user_matches[0]
    elif llm_matches:
        perimeter = llm_matches[0]
    else:
        return {
            "status": "blocked",
            "reason": "No valid IATA code found in input.",
            "perimeter": None,
            "passenger_json": "[]",
            "alarm_json": "[]",
            "messages": [HumanMessage(content="Data Agent blocked: no valid IATA code found.")]
        }

    p_df, a_df = fetch_security_context(perimeter)
    print(f"DATA AGENT: {perimeter} ({len(p_df)} records)")

    if p_df.empty:
        return {
            "status": "blocked",
            "reason": "No passenger records found for this airport.",
            "perimeter": perimeter,
            "passenger_json": "[]",
            "alarm_json": "[]",
            "messages": [HumanMessage(content=f"Data Agent blocked: no records found for {perimeter}.")]
        }

    return {
        "status": "ready",
        "perimeter": perimeter,
        "passenger_json": p_df.to_json(orient='records', date_format='iso'),
        "alarm_json": a_df.to_json(orient='records', date_format='iso'),
        "messages": [HumanMessage(content=f"Data Agent selected perimeter: {perimeter}")]
    }

from langchain_core.tools import tool


## 2. Baseline Agent

The Baseline Agent turns filtered airport data into useful analytical features.  
This is where simple rows become traffic volumes, alarm rates, rolling averages, and deviation ratios.


### 2.1 Build the baseline table

This cell builds the main analysis table similar to the classical pipeline.which is date × route_airport<br>
It calculates daily entries, investigations, alarms, rates, rolling baselines, and z-scores.

In [24]:
def build_baseline_dataframe(passenger_data: str, alarm_data: str) -> pd.DataFrame:
    p_df = pd.read_json(io.StringIO(passenger_data), orient="records")
    c_df = pd.read_json(io.StringIO(alarm_data), orient="records")

    if p_df.empty:
        return pd.DataFrame()

    # Date conversion
    p_df["date"] = pd.to_datetime(p_df["date"], errors="coerce")
    p_df = p_df.dropna(subset=["date"])

    if not c_df.empty:
        c_df["date"] = pd.to_datetime(c_df["date"], errors="coerce")
        c_df = c_df.dropna(subset=["date"])
    else:
        c_df = pd.DataFrame(columns=[
            "date", "route_airport", "route_city", "route_country",
            "event_type", "alarm_reason", "total_flights"
        ])

    # Base route-day passenger table
    route_daily = (
        p_df.groupby(["date", "route_airport"], as_index=False)
        .agg(
            route_city=("route_city", "first"),
            route_country=("route_country", "first"),
            entries=("passengers_entries_count", "sum"),
            investigations=("passengers_investigated_count", "sum"),
            flagged=("passengers_flagged_count", "sum"),
            passenger_rows=("passengers_entries_count", "size"),
            data_quality_rows=("data_quality_issue", "sum"),
            nationality_count=("nationality", "nunique"),
            document_type_count=("document_type", "nunique"),
            airline_count=("airline", "nunique"),
            control_result_count=("control_result", "nunique")
        )
    )

    # Case/alarm features
    if not c_df.empty:
        cases_route_daily = (
            c_df.groupby(["date", "route_airport"], as_index=False)
            .agg(
                case_records=("event_type", "size"),
                total_flights=("total_flights", "sum"),
                unique_alarm_reasons=("alarm_reason", "nunique"),
                unique_event_types=("event_type", "nunique")
            )
        )
    else:
        cases_route_daily = pd.DataFrame(columns=[
            "date", "route_airport", "case_records", "total_flights",
            "unique_alarm_reasons", "unique_event_types"
        ])

    route_daily = route_daily.merge(
        cases_route_daily,
        on=["date", "route_airport"],
        how="left"
    )

    fill_zero_cols = [
        "case_records", "total_flights",
        "unique_alarm_reasons", "unique_event_types"
    ]

    for col in fill_zero_cols:
        if col in route_daily.columns:
            route_daily[col] = route_daily[col].fillna(0)

    route_daily["has_case_match"] = (route_daily["case_records"] > 0).astype(int)

    # Compatibility aliases for the Report Agent
    route_daily["alarms"] = route_daily["case_records"]

    # Rates
    route_daily["investigation_rate"] = safe_rate(
        route_daily["investigations"],
        route_daily["entries"]
    )

    route_daily["flag_rate"] = safe_rate(
        route_daily["flagged"],
        route_daily["entries"]
    )

    route_daily["flag_given_investigated"] = safe_rate(
        route_daily["flagged"],
        route_daily["investigations"]
    )

    route_daily["alarm_density_per_entry"] = safe_rate(
        route_daily["case_records"],
        route_daily["entries"]
    )

    # Compatibility alias
    route_daily["alarm_rate"] = route_daily["alarm_density_per_entry"]

    # Calendar features
    route_daily["year"] = route_daily["date"].dt.year
    route_daily["month"] = route_daily["date"].dt.month
    route_daily["day"] = route_daily["date"].dt.day
    route_daily["weekday"] = route_daily["date"].dt.dayofweek
    route_daily["is_weekend"] = route_daily["weekday"].isin([5, 6]).astype(int)

    # Data-quality flags
    route_daily["zero_entries_with_cases"] = (
        (route_daily["entries"] == 0) & (route_daily["case_records"] > 0)
    )

    route_daily["invalid_rate_issue"] = (
        (route_daily["investigation_rate"] > 1)
        | (route_daily["flag_rate"] > 1)
        | (route_daily["flag_given_investigated"] > 1)
    )

    route_daily["data_quality_issue"] = (
        (route_daily["data_quality_rows"] > 0)
        | route_daily["zero_entries_with_cases"]
        | route_daily["invalid_rate_issue"]
    ).astype(int)

    # Low-volume flags
    route_daily["is_low_volume"] = (route_daily["entries"] < 10).astype(int)
    route_daily["is_low_volume_50"] = (route_daily["entries"] < 50).astype(int)

    # Sort before time-series features
    route_daily = route_daily.sort_values(["route_airport", "date"]).reset_index(drop=True)

    # Lag and change features
    change_cols = [
        "entries", "investigations", "flagged",
        "investigation_rate", "flag_rate", "alarm_density_per_entry"
    ]

    for col in change_cols:
        route_daily[col] = pd.to_numeric(route_daily[col], errors="coerce")

        lag_col = f"{col}_lag1"
        diff_col = f"{col}_diff1"
        pct_col = f"{col}_pct_change1"

        route_daily[lag_col] = route_daily.groupby("route_airport")[col].shift(1)
        route_daily[diff_col] = route_daily[col] - route_daily[lag_col]

        route_daily[pct_col] = np.nan

        valid_mask = (
            route_daily[lag_col].notna()
            & (route_daily[lag_col] != 0)
        )

        route_daily.loc[valid_mask, pct_col] = (
            route_daily.loc[valid_mask, diff_col]
            / route_daily.loc[valid_mask, lag_col]
        )

    # Rolling baselines, shifted to avoid leakage
    rolling_targets = ["entries", "flag_rate", "alarm_density_per_entry"]

    for col in rolling_targets:
        route_daily[f"{col}_roll7"] = (
            route_daily.groupby("route_airport")[col]
            .transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
        )

        route_daily[f"{col}_roll30"] = (
            route_daily.groupby("route_airport")[col]
            .transform(lambda s: s.shift(1).rolling(30, min_periods=7).mean())
        )

        route_daily[f"{col}_dev_ratio7"] = np.where(
            route_daily[f"{col}_roll7"].notna() & (route_daily[f"{col}_roll7"] > 0),
            route_daily[col] / route_daily[f"{col}_roll7"],
            np.nan
        )

    #Compatibility aliases used by older report logic
    route_daily["entries_7_mean"] = route_daily["entries_roll7"]
    route_daily["alarm_rate_7_mean"] = route_daily["alarm_density_per_entry_roll7"]
    route_daily["entries_dev_ratio"] = route_daily["entries_dev_ratio7"]
    route_daily["alarm_dev_ratio"] = route_daily["alarm_density_per_entry_dev_ratio7"]

    #Monthly seasonal baseline
    route_daily["entries_monthly_baseline"] = (
        route_daily.groupby(["route_airport", "month"])["entries"]
        .transform(lambda s: s.shift(1).expanding(min_periods=3).mean())
    )

    route_daily["entries_residual"] = (
        route_daily["entries"] - route_daily["entries_monthly_baseline"]
    )

    route_daily["entries_residual_std"] = (
        route_daily.groupby("route_airport")["entries_residual"]
        .transform(lambda s: s.shift(1).expanding(min_periods=7).std())
    )

    route_daily["entries_residual_z"] = np.where(
        route_daily["entries_residual_std"].notna() & (route_daily["entries_residual_std"] > 0),
        route_daily["entries_residual"] / route_daily["entries_residual_std"],
        np.nan
    )

    route_daily["entries_route_mean"] = (
        route_daily.groupby("route_airport")["entries"]
        .transform(lambda s: s.shift(1).expanding(min_periods=7).mean())
    )

    route_daily["entries_vs_route_mean"] = np.where(
        route_daily["entries_route_mean"] > 0,
        route_daily["entries"] / route_daily["entries_route_mean"],
        np.nan
    )

    #Compatibility alias
    route_daily["entries_z_score"] = route_daily["entries_residual_z"]

    # Missing value handling, similar to classical pipeline
    numeric_cols = route_daily.select_dtypes(include=[np.number]).columns
    route_daily[numeric_cols] = route_daily[numeric_cols].replace([np.inf, -np.inf], np.nan)

    for col in numeric_cols:
        if route_daily[col].isna().all():
            route_daily[col] = 0
        else:
            route_daily[col] = route_daily[col].fillna(route_daily[col].median())

    return route_daily


def calculate_security_stats(passenger_data: str, alarm_data: str):
    """
    Compact statistics used for logging or optional agent summaries.
    """
    baseline_df = build_baseline_dataframe(passenger_data, alarm_data)

    if baseline_df.empty:
        return {
            "total_route_day_rows": 0,
            "total_entries": 0,
            "total_cases": 0,
            "mean_alarm_density": 0.0,
            "data_quality_rows": 0
        }

    return {
        "total_route_day_rows": int(len(baseline_df)),
        "total_entries": int(baseline_df["entries"].sum()),
        "total_cases": int(baseline_df["case_records"].sum()),
        "mean_alarm_density": round(float(baseline_df["alarm_density_per_entry"].mean()), 4),
        "data_quality_rows": int(baseline_df["data_quality_issue"].sum())
    }

### 2.2 Baseline Agent Node
The Baseline Agent transforms the filtered data into an analytical table. Here we create route-level features such as entries, investigations, alarms, rates, rolling averages, deviations, and data-quality flags.

In [25]:
@tool
def security_math_tool(passenger_json: str, alarm_json: str):
    """Use this tool to calculate precise security statistics and anomaly thresholds."""
    return calculate_security_stats(passenger_json, alarm_json)

def baseline_agent_node(state: AgentState):
    print("BASELINE AGENT: Preparing Data Hand-off for Outlier Agent")

    baseline_df = build_baseline_dataframe(state["passenger_json"], state["alarm_json"])
    baseline_dataframe_json = baseline_df.to_json(orient='records', date_format='iso')

    report_msg = HumanMessage(
        content=(
            f"Baseline dataframe generated for {state['perimeter']} "
            f"with {len(baseline_df)} daily rows. Ready for Outlier Detection."
        )
    )

    return {
        "baseline_dataframe_json": baseline_dataframe_json,
        "messages": [report_msg]
    }


## X. Data Integrity Check


### X.1 Available Airport/City Check
Before running the agents, we quickly check which airport codes and cities exist in the cleaned data.  
This avoids running the workflow on a perimeter that is not available in the dataset.

In [26]:
if PASSENGER_CLEAN_PATH.exists():
    df_audit = pd.read_csv(PASSENGER_CLEAN_PATH)

    print("Available arrival airport codes:")
    print(df_audit["arrival_airport_code"].unique())
else:
    print("Clean passenger file not found.")
    print("Run the classical preprocessing cells first.")

Available arrival airport codes:
<StringArray>
['NAP', 'TSF', 'FCO', 'BGY', 'PSA', 'BLQ', 'MXP', 'LIN', 'VRN', 'BRI', 'CTA',
 'PMO', 'CIA', 'VCE', 'PEG', 'CAG', 'GOA', 'FLR', 'TRN', 'AOI', 'PSR', 'CIY',
 'TRS', 'BDS', 'RMI', 'CUF', 'SUF', 'PMF', 'REG']
Length: 29, dtype: str


### X.2 Validate the baseline output

This function checks whether the baseline table is complete and numerically valid.  
It helps catch missing columns, negative counts, or suspicious data-quality issues before trusting the anomaly results.

In [27]:
def validate_baseline_output(passenger_json: str, alarm_json: str, verbose: bool = True):
    """
    Validates the route-level baseline dataframe before detection.
    """

    print("=== BUILDING BASELINE DATAFRAME ===")
    baseline_df = build_baseline_dataframe(passenger_json, alarm_json)

    print("\n[1] BASIC STRUCTURE")
    print("Shape:", baseline_df.shape)
    print("Columns:", list(baseline_df.columns))
    display(baseline_df.head())

    assert not baseline_df.empty, "Baseline dataframe is empty."

    required_cols = [
        "date",
        "route_airport",
        "route_city",
        "route_country",
        "entries",
        "investigations",
        "flagged",
        "case_records",
        "alarm_density_per_entry",
        "investigation_rate",
        "flag_rate",
        "entries_roll7",
        "entries_dev_ratio7",
        "entries_residual_z",
        "consensus_ready_placeholder"
    ]

    # This placeholder is only for checking logic below
    required_cols.remove("consensus_ready_placeholder")

    missing_cols = [col for col in required_cols if col not in baseline_df.columns]
    assert len(missing_cols) == 0, f"Missing required columns: {missing_cols}"

    print("Required columns check: OK")

    print("\n[2] NUMERIC SANITY CHECKS")

    non_negative_cols = [
        "entries",
        "investigations",
        "flagged",
        "case_records",
        "total_flights"
    ]

    for col in non_negative_cols:
        if col in baseline_df.columns:
            assert (baseline_df[col] >= 0).all(), f"Negative values found in {col}"

    print("Non-negative count checks: OK")

    print("\n[3] DATA-QUALITY WARNINGS")
    dq_rows = baseline_df[baseline_df["data_quality_issue"] == 1]

    print("Rows with data-quality issues:", len(dq_rows))

    if len(dq_rows) > 0:
        display(dq_rows[[
            "date", "route_airport", "entries", "investigations",
            "flagged", "case_records", "zero_entries_with_cases",
            "invalid_rate_issue"
        ]].head(10))

    print("\n[4] JSON SERIALIZATION TEST")
    baseline_dataframe_json = baseline_df.to_json(orient="records", date_format="iso")
    reconstructed_df = pd.read_json(io.StringIO(baseline_dataframe_json), orient="records")

    assert reconstructed_df.shape[0] == baseline_df.shape[0], "Row count changed after serialization."
    assert set(reconstructed_df.columns) == set(baseline_df.columns), "Columns changed after serialization."

    print("Serialization/deserialization check: OK")

    return {
        "rows": len(baseline_df),
        "columns": list(baseline_df.columns),
        "data_quality_rows": len(dq_rows),
        "status": "valid"
    }

## 3. Outlier Detection

The Agent (`phi4-mini-reasoning`) looks for unusual route-day patterns. It reads the baseline dataframe, runs three unsupervised methods (IsolationForest, LOF, Z-score), takes a consensus of at least 2 out of 3, and lets the LLM interpret the flagged rows.

In [28]:
DETECTION_FEATURES = [
    "entries",
    "entries_lag1",
    "entries_diff1",
    "entries_pct_change1",
    "entries_roll7",
    "entries_roll30",
    "entries_dev_ratio7",
    "entries_monthly_baseline",
    "entries_residual",
    "entries_residual_z",
    "entries_route_mean",
    "entries_vs_route_mean",
    "weekday",
    "is_weekend",
    "is_low_volume",
    "is_low_volume_50",
    "flag_rate",
    "flag_rate_roll7",
    "flag_rate_dev_ratio7",
    "alarm_density_per_entry",
    "alarm_density_per_entry_roll7",
    "alarm_density_per_entry_dev_ratio7",
    "nationality_count",
    "document_type_count",
    "airline_count",
    "control_result_count",
    "case_records",
    "unique_alarm_reasons",
    "unique_event_types"
]


def prepare_feature_matrix(df, feature_cols):
    """
    Same principle as the classical pipeline:
    - select numeric detection features
    - replace infinities
    - fill missing values with median
    - fit scaler on earlier 70% of dates
    """

    feature_cols = [c for c in feature_cols if c in df.columns]

    X = df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)

    for col in X.columns:
        if X[col].isna().all():
            X[col] = 0
        else:
            X[col] = X[col].fillna(X[col].median())

    scaler = StandardScaler()

    cutoff_date = df["date"].quantile(0.7)
    train_mask = df["date"] <= cutoff_date
    test_mask = df["date"] > cutoff_date

    # Fallback for small or strange samples
    if train_mask.sum() < 5 or test_mask.sum() < 1:
        train_mask = pd.Series(True, index=df.index)
        test_mask = pd.Series(False, index=df.index)

    X_scaled = np.empty_like(X.values, dtype=float)

    X_train_scaled = scaler.fit_transform(X.loc[train_mask])
    X_scaled[train_mask.values] = X_train_scaled

    if test_mask.sum() > 0:
        X_test_scaled = scaler.transform(X.loc[test_mask])
        X_scaled[test_mask.values] = X_test_scaled

    return X, X_scaled, train_mask, test_mask, feature_cols


def run_isolation_forest_classical_style(df, X_scaled, train_mask, test_mask):
    iso = IsolationForest(
        contamination=CONTAMINATION,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    iso.fit(X_scaled[train_mask.values])

    labels = np.zeros(len(df))
    scores = np.zeros(len(df))

    labels[train_mask.values] = iso.predict(X_scaled[train_mask.values])
    scores[train_mask.values] = iso.decision_function(X_scaled[train_mask.values])

    if test_mask.sum() > 0:
        labels[test_mask.values] = iso.predict(X_scaled[test_mask.values])
        scores[test_mask.values] = iso.decision_function(X_scaled[test_mask.values])

    return labels, scores


def run_lof_classical_style(df, X_scaled, train_mask, test_mask):
    n_neighbors = min(20, max(2, train_mask.sum() - 1))

    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=CONTAMINATION,
        novelty=True
    )

    lof.fit(X_scaled[train_mask.values])

    labels = np.zeros(len(df))
    scores = np.zeros(len(df))

    labels[train_mask.values] = lof.predict(X_scaled[train_mask.values])
    scores[train_mask.values] = -lof.score_samples(X_scaled[train_mask.values])

    if test_mask.sum() > 0:
        labels[test_mask.values] = lof.predict(X_scaled[test_mask.values])
        scores[test_mask.values] = -lof.score_samples(X_scaled[test_mask.values])

    return labels, scores


def run_zscore_classical_style(df, X_scaled, train_mask):
    zscore_max = np.abs(X_scaled).max(axis=1)

    if train_mask.sum() > 0:
        threshold = np.quantile(zscore_max[train_mask.values], 0.95)
    else:
        threshold = np.quantile(zscore_max, 0.95)

    z_labels = (zscore_max > threshold).astype(int)

    return z_labels, zscore_max, threshold


def apply_business_rules(df):
    """
    Classical-style post-processing rules.
    These rules make the system operationally interpretable.
    """

    df["rule_entries_spike"] = (df["entries_dev_ratio7"] > 3).astype(int)
    df["rule_high_residual_z"] = (df["entries_residual_z"].abs() > 2).astype(int)
    df["rule_alarm_density_spike"] = (
        df["alarm_density_per_entry_dev_ratio7"] > 3
    ).astype(int)
    df["rule_high_flag_rate"] = (df["flag_rate"] > 0.5).astype(int)
    df["rule_data_quality_issue"] = (df["data_quality_issue"] == 1).astype(int)

    rule_cols = [
        "rule_entries_spike",
        "rule_high_residual_z",
        "rule_alarm_density_spike",
        "rule_high_flag_rate",
        "rule_data_quality_issue"
    ]

    df["rule_count"] = df[rule_cols].sum(axis=1)
    df["rule_any"] = (df["rule_count"] > 0).astype(int)

    return df, rule_cols


def outlier_detection_agent_node(state):
    print("OUTLIER DETECTION AGENT")

    df = pd.read_json(io.StringIO(state["baseline_dataframe_json"]), orient="records")
    df["date"] = pd.to_datetime(df["date"])

    df = df.sort_values(["date", "route_airport"]).reset_index(drop=True)

    if len(df) < MIN_ROWS_FOR_DETECTION:
        result = {"error": f"Only {len(df)} rows — not enough for detection."}
        return {
            "anomaly_results": json.dumps(result),
            "scored_dataframe_json": "[]",
            "messages": [HumanMessage(content=f"Outlier Detection skipped: {len(df)} rows.")]
        }

    X, X_scaled, train_mask, test_mask, feature_cols = prepare_feature_matrix(
        df,
        DETECTION_FEATURES
    )

    # Classical detectors
    iso_labels, iso_scores = run_isolation_forest_classical_style(
        df, X_scaled, train_mask, test_mask
    )

    lof_labels, lof_scores = run_lof_classical_style(
        df, X_scaled, train_mask, test_mask
    )

    z_labels, z_scores, z_threshold = run_zscore_classical_style(
        df, X_scaled, train_mask
    )

    df["iso_anomaly"] = (iso_labels == -1).astype(int)
    df["iso_score"] = iso_scores

    df["lof_anomaly"] = (lof_labels == -1).astype(int)
    df["lof_score"] = lof_scores

    df["zscore_anomaly"] = z_labels
    df["zscore_max"] = z_scores

    # Consensus voting
    df["anomaly_votes"] = (
        df["iso_anomaly"] +
        df["lof_anomaly"] +
        df["zscore_anomaly"]
    )

    df["consensus_anomaly"] = (df["anomaly_votes"] >= 2).astype(int)

    # Business rules
    df, rule_cols = apply_business_rules(df)

    # Classical final anomaly logic
    df["final_anomaly"] = (
        (df["consensus_anomaly"] == 1) |
        (df["rule_any"] == 1)
    ).astype(int)

    # Human-readable evidence type
    def evidence_type(row):
        model_hit = row["consensus_anomaly"] == 1
        rule_hit = row["rule_any"] == 1

        if model_hit and rule_hit:
            return "Model + rules"
        if model_hit:
            return "Model consensus only"
        if rule_hit:
            return "Business rules only"
        return "Not flagged"

    df["evidence_type"] = df.apply(evidence_type, axis=1)

    # Compact flagged output
    flagged_df = df[df["final_anomaly"] == 1].copy()

    flagged_rows = []

    for _, row in flagged_df.iterrows():
        methods = []

        if row["iso_anomaly"] == 1:
            methods.append("IsolationForest")
        if row["lof_anomaly"] == 1:
            methods.append("LOF")
        if row["zscore_anomaly"] == 1:
            methods.append("Z-score")

        triggered_rules = [c for c in rule_cols if row[c] == 1]

        flagged_rows.append({
            "date": str(row["date"].date()),
            "route_airport": row["route_airport"],
            "route_city": row["route_city"],
            "entries": int(row["entries"]),
            "investigations": int(row["investigations"]),
            "flagged": int(row["flagged"]),
            "case_records": int(row["case_records"]),
            "alarm_density_per_entry": round(float(row["alarm_density_per_entry"]), 4),
            "flag_rate": round(float(row["flag_rate"]), 4),
            "entries_dev_ratio7": round(float(row["entries_dev_ratio7"]), 2),
            "entries_residual_z": round(float(row["entries_residual_z"]), 2),
            "alarm_density_dev_ratio7": round(float(row["alarm_density_per_entry_dev_ratio7"]), 2),
            "anomaly_votes": int(row["anomaly_votes"]),
            "flagged_by": methods,
            "triggered_rules": triggered_rules,
            "evidence_type": row["evidence_type"],
            "data_quality_issue": bool(row["data_quality_issue"])
        })

    print(
        f"IF: {df['iso_anomaly'].sum()}, "
        f"LOF: {df['lof_anomaly'].sum()}, "
        f"Z: {df['zscore_anomaly'].sum()}, "
        f"Consensus: {df['consensus_anomaly'].sum()}, "
        f"Final anomalies: {df['final_anomaly'].sum()}"
    )

    # LLM analytical note
    llm = ChatOllama(model="phi4-mini-reasoning", temperature=0)

    prompt = f"""
Airport: {state["perimeter"]}

The Multi-Agent system analyzed {len(df)} route-day observations.

Classical-style anomaly detection was applied:
- Isolation Forest
- Local Outlier Factor
- Z-score
- consensus voting
- business-rule post-processing

Final flagged rows:
{json.dumps(flagged_rows[:20], indent=2)}

Write a short analytical note of maximum 6 sentences.
Mention:
- whether anomalies are mostly model-driven, rule-driven, or both
- whether they are caused by route volume, alarm/case density, or data-quality issues
- the most important dates/routes

Do not invent numbers.
Do not mention suspicious nationalities or groups.
"""

    analysis = clean_llm_output(llm.invoke([HumanMessage(content=prompt)]).content)

    result = {
        "airport": state["perimeter"],
        "total_rows": int(len(df)),
        "feature_cols": feature_cols,
        "method_counts": {
            "IsolationForest": int(df["iso_anomaly"].sum()),
            "LOF": int(df["lof_anomaly"].sum()),
            "Z-score": int(df["zscore_anomaly"].sum())
        },
        "consensus_count": int(df["consensus_anomaly"].sum()),
        "business_rule_count": int(df["rule_any"].sum()),
        "final_anomaly_count": int(df["final_anomaly"].sum()),
        "zscore_threshold": round(float(z_threshold), 4),
        "flagged_rows": flagged_rows,
        "analysis": analysis
    }

    return {
        "anomaly_results": json.dumps(result),
        "scored_dataframe_json": df.to_json(orient="records", date_format="iso"),
        "messages": [
            HumanMessage(
                content=f"Outlier Detection completed: {result['final_anomaly_count']} final anomalies."
            )
        ]
    }

## 4. Risk Profiling Agent

The Agent (`qwen2.5:7b`) gives meaning to the detected anomalies, evaluates each flagged observation against four risk indicators and assigns a risk level (CRITICAL, HIGH, MODERATE, LOW) with a one-sentence rationale.

In [29]:
def risk_profiling_agent_node(state):
    print("RISK PROFILING AGENT")

    anomaly_data = json.loads(state["anomaly_results"])

    if "error" in anomaly_data:
        return {
            "risk_profile": json.dumps({"error": anomaly_data["error"]}),
            "messages": [HumanMessage(content="Risk profiling skipped.")]
        }

    df = pd.read_json(io.StringIO(state["scored_dataframe_json"]), orient="records")
    df["date"] = pd.to_datetime(df["date"])

    if df.empty:
        return {
            "risk_profile": json.dumps({"airport": state["perimeter"], "assessments": []}),
            "messages": [HumanMessage(content="No rows to assess.")]
        }

    model_cols = ["iso_anomaly", "lof_anomaly", "zscore_anomaly"]
    rule_cols = [
        "rule_entries_spike",
        "rule_high_residual_z",
        "rule_alarm_density_spike",
        "rule_high_flag_rate",
        "rule_data_quality_issue"
    ]

    df["model_agreement"] = df[model_cols].sum(axis=1)

    score_components = pd.DataFrame(index=df.index)
    score_components["model_agreement"] = (df["model_agreement"] / len(model_cols)).clip(0, 1)
    score_components["rule_strength"] = (df["rule_count"] / len(rule_cols)).clip(0, 1)
    score_components["entries_spike"] = robust_score(df["entries_dev_ratio7"])
    score_components["seasonal_residual"] = robust_score(df["entries_residual_z"].abs())
    score_components["alarm_density"] = robust_score(df["alarm_density_per_entry_dev_ratio7"])

    weights = {
        "model_agreement": 0.35,
        "rule_strength": 0.25,
        "entries_spike": 0.15,
        "seasonal_residual": 0.15,
        "alarm_density": 0.10
    }

    df["risk_score"] = sum(
        score_components[col] * weight
        for col, weight in weights.items()
    ) * 100

    df.loc[df["final_anomaly"] == 0, "risk_score"] *= 0.25
    df["risk_score"] = df["risk_score"].round(2)

    df["risk_level"] = pd.cut(
        df["risk_score"],
        bins=[-0.01, 20, 40, 60, 100],
        labels=["LOW", "MODERATE", "HIGH", "CRITICAL"]
    ).astype(str)

    df["review_priority"] = "Not flagged"

    flagged_mask = df["final_anomaly"] == 1

    if flagged_mask.sum() > 0:
        flagged_scores = df.loc[flagged_mask, "risk_score"]
        q50, q80, q95 = flagged_scores.quantile([0.50, 0.80, 0.95])

        def assign_priority(score):
            if score >= q95:
                return "P1 - immediate review"
            if score >= q80:
                return "P2 - high priority"
            if score >= q50:
                return "P3 - standard review"
            return "P4 - monitor"

        df.loc[flagged_mask, "review_priority"] = flagged_scores.apply(assign_priority)

    assessments = []

    for _, row in df.loc[flagged_mask].sort_values("risk_score", ascending=False).iterrows():
        triggered = []

        for c in rule_cols:
            if row.get(c, 0) == 1:
                triggered.append(c)

        if row["consensus_anomaly"] == 1:
            triggered.append("model_consensus")

        note_parts = []

        if row["data_quality_issue"] == 1:
            note_parts.append("contains a data-quality warning")

        if row["rule_alarm_density_spike"] == 1:
            note_parts.append("alarm/case density is above baseline")

        if row["rule_entries_spike"] == 1:
            note_parts.append("entries are above recent route baseline")

        if row["rule_high_residual_z"] == 1:
            note_parts.append("seasonal residual is unusually high")

        if not note_parts:
            note_parts.append("flagged mainly by model agreement")

        assessments.append({
            "date": str(row["date"].date()),
            "route_airport": row["route_airport"],
            "route_city": row["route_city"],
            "risk_score": float(row["risk_score"]),
            "risk_level": row["risk_level"],
            "review_priority": row["review_priority"],
            "evidence_type": row["evidence_type"],
            "triggered": triggered,
            "note": "; ".join(note_parts) + "."
        })

    distribution = (
        pd.Series([a["risk_level"] for a in assessments])
        .value_counts()
        .to_dict()
    )

    profile = {
        "airport": state["perimeter"],
        "total_assessed": len(assessments),
        "distribution": distribution,
        "assessments": assessments
    }

    print(f"Risk distribution: {distribution}")

    return {
        "risk_profile": json.dumps(profile),
        "scored_dataframe_json": df.to_json(orient="records", date_format="iso"),
        "messages": [HumanMessage(content=f"Risk profile completed: {distribution}")]
    }

## 5. Report Agent
The Report Agent (`llama3.2:3b`) prepares the final explanation. First, this function combines anomaly results and risk results into one clean payload so the report stays consistent.


### 5.1 Report Payload
We combine anomaly results and risk scores into one clean structure.  
This prevents the Report Agent from inventing or mismatching numbers.

In [30]:

def build_report_payload(state: AgentState):
    """
    Builds a structured, comparison-ready payload for the Report Agent.

    Important:
    The LLM should not invent report facts. Therefore, all metrics, routes,
    priorities, and anomaly details are computed here first in Python.
    """

    anomaly_data = json.loads(state["anomaly_results"])
    risk_data = json.loads(state["risk_profile"])

    if "error" in anomaly_data:
        return {
            "airport": state["perimeter"],
            "status": "error",
            "message": anomaly_data["error"]
        }

    if "error" in risk_data:
        return {
            "airport": state["perimeter"],
            "status": "error",
            "message": risk_data["error"]
        }

    df = pd.read_json(io.StringIO(state["scored_dataframe_json"]), orient="records")

    if df.empty:
        return {
            "airport": state["perimeter"],
            "status": "error",
            "message": "No scored rows are available for the selected airport."
        }

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    airport = state["perimeter"].upper().strip()
    df["arrival_airport"] = airport

    flagged_df = (
        df.loc[df["final_anomaly"] == 1]
        .sort_values(["risk_score", "model_agreement", "rule_count"], ascending=False)
        .reset_index(drop=True)
    )

    summary_overall = {
        "airport": airport,
        "total_observations": int(len(df)),
        "final_anomalies": int(df["final_anomaly"].sum()),
        "final_anomaly_rate": round(float(df["final_anomaly"].mean()), 4),
        "model_consensus_anomalies": int(df["consensus_anomaly"].sum()),
        "business_rule_anomalies": int(df["rule_any"].sum()),
        "model_and_rule_overlap": int(((df["consensus_anomaly"] == 1) & (df["rule_any"] == 1)).sum()),
        "average_risk_score_all_rows": round(float(df["risk_score"].mean()), 2),
        "average_risk_score_anomalies": (
            round(float(flagged_df["risk_score"].mean()), 2)
            if len(flagged_df) > 0 else 0
        ),
    }

    priority_distribution = (
        flagged_df.groupby("review_priority")
        .agg(
            anomalies=("final_anomaly", "size"),
            avg_risk_score=("risk_score", "mean")
        )
        .reset_index()
        .sort_values("avg_risk_score", ascending=False)
        .round(4)
        .to_dict(orient="records")
        if len(flagged_df) > 0 else []
    )

    evidence_distribution = (
        flagged_df.groupby("evidence_type")
        .agg(
            anomalies=("final_anomaly", "size"),
            avg_risk_score=("risk_score", "mean")
        )
        .assign(share_of_anomalies=lambda x: x["anomalies"] / max(x["anomalies"].sum(), 1))
        .reset_index()
        .sort_values("anomalies", ascending=False)
        .round(4)
        .to_dict(orient="records")
        if len(flagged_df) > 0 else []
    )

    top_routes = (
        df.groupby("route_city")
        .agg(
            observations=("final_anomaly", "size"),
            anomalies=("final_anomaly", "sum"),
            avg_risk_score=("risk_score", "mean"),
            max_risk_score=("risk_score", "max")
        )
        .assign(anomaly_rate=lambda x: x["anomalies"] / x["observations"])
        .query("anomalies > 0")
        .sort_values(["anomalies", "max_risk_score"], ascending=False)
        .head(10)
        .reset_index()
        .round(4)
        .to_dict(orient="records")
    )

    technical_cols = [
        "date",
        "arrival_airport",
        "route_airport",
        "route_city",
        "entries",
        "entries_roll7",
        "entries_dev_ratio7",
        "entries_monthly_baseline",
        "entries_residual",
        "entries_residual_z",
        "investigations",
        "flagged",
        "flag_rate",
        "case_records",
        "alarm_density_per_entry",
        "iso_anomaly",
        "lof_anomaly",
        "zscore_anomaly",
        "anomaly_votes",
        "consensus_anomaly",
        "rule_entries_spike",
        "rule_high_residual_z",
        "rule_alarm_density_spike",
        "rule_high_flag_rate",
        "rule_data_quality_issue",
        "rule_count",
        "rule_any",
        "final_anomaly",
        "evidence_type",
        "risk_score",
        "risk_level",
        "review_priority",
    ]

    technical_cols = [c for c in technical_cols if c in flagged_df.columns]

    top_anomalies = (
        flagged_df[technical_cols]
        .head(10)
        .copy()
    )

    if "date" in top_anomalies.columns:
        top_anomalies["date"] = pd.to_datetime(top_anomalies["date"]).dt.strftime("%Y-%m-%d")

    top_anomalies = top_anomalies.round(4).to_dict(orient="records")

    return {
        "airport": airport,
        "status": "ok",
        "summary_overall": summary_overall,
        "priority_distribution": priority_distribution,
        "evidence_distribution": evidence_distribution,
        "top_routes": top_routes,
        "top_anomalies": top_anomalies,
        "method_counts": anomaly_data.get("method_counts"),
        "feature_cols": anomaly_data.get("feature_cols"),
        "zscore_threshold": anomaly_data.get("zscore_threshold"),
        "anomaly_analysis": anomaly_data.get("analysis"),
    }


### 5.2 Report structure

This node asks the LLM to write the Transit Anomaly Report.  
The LLM does not create new numbers; it only explains the structured results produced by the previous agents.

In [31]:

def _records_to_markdown(records, empty_message="No data available."):
    """Small helper to turn structured records into a Markdown table."""
    if not records:
        return empty_message
    return pd.DataFrame(records).to_markdown(index=False)


def _summary_dict_to_markdown(summary_dict):
    """Formats one summary dictionary as a two-column Markdown table."""
    return pd.DataFrame({
        "metric": list(summary_dict.keys()),
        "value": list(summary_dict.values())
    }).to_markdown(index=False)


def report_agent_node(state: AgentState):
    print("REPORT AGENT")

    payload = build_report_payload(state)

    if payload["status"] == "error":
        report = f"""# Transit Anomaly Report — {payload["airport"]}

## Status
The report could not be generated because the previous step returned an error.

## Error Message
{payload["message"]}

## Interpretation
This is not necessarily a system failure. In most cases, it means that the selected airport does not have enough route-day observations for reliable anomaly detection.
"""
        return {
            "final_report": report,
            "messages": [HumanMessage(content="Report generation failed due to previous error.")]
        }

    # The LLM is used only for a short interpretation note.
    # All tables and metrics are created deterministically from the structured data.
    interpretation_note = payload.get("anomaly_analysis", "")

    try:
        llm = ChatOllama(model="llama3.2:3b", temperature=0)

        llm_prompt = f"""
You are the Report Agent in a Multi-Agent System for airport anomaly detection.

Write ONLY one short technical interpretation paragraph, maximum 5 sentences.

Use only these facts:
{json.dumps({
    "airport": payload["airport"],
    "summary_overall": payload["summary_overall"],
    "top_routes": payload["top_routes"][:5],
    "top_anomalies": payload["top_anomalies"][:5],
    "method_counts": payload["method_counts"],
}, indent=2)}

Rules:
- Do not invent causes.
- Do not mention booking practices.
- Do not mention airlines unless an airline column/value is explicitly provided.
- Do not mention SPC charts.
- Do not describe any nationality, city, or group as suspicious by itself.
- Focus on route-day traffic deviations, model agreement, business-rule evidence, risk score, and review priority.
"""
        interpretation_note = clean_llm_output(
            llm.invoke([HumanMessage(content=llm_prompt)]).content
        )

    except Exception as e:
        interpretation_note = (
            "The report uses structured anomaly evidence only. "
            "The selected airport contains route-day records flagged by a mix of model consensus and business rules. "
            "The ranked cases should be reviewed as an investigation queue, not as confirmed security incidents."
        )

    report = f"""# Transit Anomaly Report — {payload["airport"]}

## Executive Summary

The Multi-Agent pipeline analyzed **{payload["summary_overall"]["total_observations"]:,} route-day observations** for arrival airport **{payload["airport"]}** and flagged **{payload["summary_overall"]["final_anomalies"]:,} final anomalies**.

This report is generated from the structured outputs of the Data Agent, Baseline Agent, Outlier Detection Agent, and Risk Profiling Agent. The LLM is used only to turn the structured results into readable language.

## Detection Methods

The pipeline uses the following logic:

1. **Data Agent** filters the passenger and case/alarm data for the selected arrival airport.
2. **Baseline Agent** builds route-day features such as entries, rates, rolling baselines, residuals, and case/alarm density.
3. **Outlier Detection Agent** applies Isolation Forest, LOF, and Z-score.
4. **Consensus logic** flags observations where at least two models agree.
5. **Business rules** flag operational spikes such as entries above baseline or unusually high residuals.
6. **Risk Profiling Agent** assigns risk scores, risk levels, and review priorities.

## Main Metrics

{_summary_dict_to_markdown(payload["summary_overall"])}

## Detector Counts

{_summary_dict_to_markdown(payload["method_counts"])}

## Review-Priority Distribution

{_records_to_markdown(payload["priority_distribution"], "No review-priority distribution available because no anomalies were flagged.")}

## Evidence-Type Distribution

{_records_to_markdown(payload["evidence_distribution"], "No evidence-type distribution available because no anomalies were flagged.")}

## Top Anomalous Routes

{_records_to_markdown(payload["top_routes"], "No anomalous routes were detected.")}

## Top 10 Ranked Anomalies

{_records_to_markdown(payload["top_anomalies"], "No final anomalies were detected.")}

## Technical Interpretation

{interpretation_note}

## Recommended Follow-up Actions

- Start from `P1 - immediate review` cases, then review `P2` and `P3`.
- Check whether high-risk records are supported by both model consensus and business rules.
- Treat `Business rules only` cases as useful monitoring alerts, but validate them carefully because they may include more false positives.
- Do not treat the output as confirmed suspicious behavior. It is an alert-prioritization system.

## Limitations

- There are no ground-truth anomaly labels, so precision, recall, F1-score, and ROC-AUC cannot be measured honestly.
- Small airports may not have enough observations for reliable detection.
- The report explains statistical deviations, not confirmed operational incidents.
- Expert validation is needed before taking real-world action.
"""

    return {
        "final_report": report,
        "messages": [HumanMessage(content="Final technical Transit Anomaly Report generated.")]
    }


## 6. The Workflow Graph
Now we connect all agents into one pipeline. <br><br>
Using `StateGraph`, we define the "brain" of the operation. The workflow is a Directed Acyclic Graph (DAG) that ensures the **Data Agent** completes its task before passing the context to the **Baseline Agent**. We also integrate `MemorySaver` to allow for threaded conversations.

In [32]:
builder = StateGraph(AgentState)

builder.add_node("data_agent", data_agent_node)
builder.add_node("baseline_agent", baseline_agent_node)
builder.add_node("outlier_detection_agent", outlier_detection_agent_node)
builder.add_node("risk_profiling_agent", risk_profiling_agent_node)
builder.add_node("report_agent", report_agent_node)

builder.set_entry_point("data_agent")

builder.add_edge("data_agent", "baseline_agent")
builder.add_edge("baseline_agent", "outlier_detection_agent")
builder.add_edge("outlier_detection_agent", "risk_profiling_agent")
builder.add_edge("risk_profiling_agent", "report_agent")
builder.add_edge("report_agent", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)



## 7. Execution and Output
This is the final execution block. The user identifies a perimeter, and the MAS generates a finalized report. <br><br> 
**Note**: *If you see a 100% alarm rate for low-volume perimeters (like LIN), the agent correctly identifies this as a "low entries" incident rather than a systemic failure, demonstrating the power of combined math and reasoning.*

### 7.1 Perimeter Input

In [33]:
user_q = input("Identify target perimeter, e.g. FCO, MXP, LIN, NAP: ")

if user_q:
    config = {"configurable": {"thread_id": "v2"}}
    final_state = None

    for event in app.stream(
        {"messages": [HumanMessage(content=user_q)]},
        config,
        stream_mode="values"
    ):
        final_state = event

    print("\n" + "=" * 70)
    print(f"PERIMETER: {final_state.get('perimeter')}")
    print("=" * 70)

    print("\n--- OUTLIER DETECTION RESULTS ---")
    anomaly = json.loads(final_state["anomaly_results"])

    if "error" in anomaly:
        print(f"Error: {anomaly['error']}")
    else:
        print(f"Total route-day rows analyzed: {anomaly['total_rows']}")
        print(f"Method counts: {anomaly['method_counts']}")
        print(f"Consensus anomalies: {anomaly['consensus_count']}")
        print(f"Business-rule anomalies: {anomaly['business_rule_count']}")
        print(f"Final anomalies: {anomaly['final_anomaly_count']}")
        print(f"\nAnalytical note:\n{anomaly['analysis']}")

    print("\n--- RISK PROFILING RESULTS ---")
    risk = json.loads(final_state["risk_profile"])

    if "error" in risk:
        print(f"Error: {risk['error']}")
    else:
        print(f"Total assessed: {risk['total_assessed']}")
        print(f"Distribution: {risk['distribution']}")

        print("\nTop assessments:")
        for a in risk["assessments"][:10]:
            print(json.dumps(a, indent=2))

DATA AGENT: FCO
Passenger rows selected: 783
Case rows selected: 1272
DATA AGENT: FCO (783 records)
BASELINE AGENT: Preparing Data Hand-off for Outlier Agent
OUTLIER DETECTION AGENT
IF: 35, LOF: 27, Z: 33, Consensus: 19, Final anomalies: 130
RISK PROFILING AGENT
Risk distribution: {'LOW': 81, 'MODERATE': 42, 'HIGH': 5, 'CRITICAL': 2}
REPORT AGENT

PERIMETER: FCO

--- OUTLIER DETECTION RESULTS ---
Total route-day rows analyzed: 579
Method counts: {'IsolationForest': 35, 'LOF': 27, 'Z-score': 33}
Consensus anomalies: 19
Business-rule anomalies: 118
Final anomalies: 130

Analytical note:
The dataset reveals that anomalies at FCO are driven by both **model-based** (e.g., IsolationForest) and **rule-based systems** (e.g., "rule_high_flag_rate" or "rule_alarm_density_spike"). Anomalies often stem from high route activity, such as unusually large volumes of entries/investigations triggering statistical outliers in model-driven flags. Specific spikes in alarm/case density also activate rule-ba

### 7.2 Validate selected airport
Here we validate the baseline created for the airport we actually selected.  
It is useful as a final check before presenting or saving the results.

In [34]:
validation_result = validate_baseline_output(
    passenger_json=final_state["passenger_json"],
    alarm_json=final_state["alarm_json"]
)

=== BUILDING BASELINE DATAFRAME ===

[1] BASIC STRUCTURE
Shape: (579, 72)
Columns: ['date', 'route_airport', 'route_city', 'route_country', 'entries', 'investigations', 'flagged', 'passenger_rows', 'data_quality_rows', 'nationality_count', 'document_type_count', 'airline_count', 'control_result_count', 'case_records', 'total_flights', 'unique_alarm_reasons', 'unique_event_types', 'has_case_match', 'alarms', 'investigation_rate', 'flag_rate', 'flag_given_investigated', 'alarm_density_per_entry', 'alarm_rate', 'year', 'month', 'day', 'weekday', 'is_weekend', 'zero_entries_with_cases', 'invalid_rate_issue', 'data_quality_issue', 'is_low_volume', 'is_low_volume_50', 'entries_lag1', 'entries_diff1', 'entries_pct_change1', 'investigations_lag1', 'investigations_diff1', 'investigations_pct_change1', 'flagged_lag1', 'flagged_diff1', 'flagged_pct_change1', 'investigation_rate_lag1', 'investigation_rate_diff1', 'investigation_rate_pct_change1', 'flag_rate_lag1', 'flag_rate_diff1', 'flag_rate_pct

,date,route_airport,route_city,route_country,entries,investigations,flagged,passenger_rows,data_quality_rows,nationality_count,...,alarm_rate_7_mean,entries_dev_ratio,alarm_dev_ratio,entries_monthly_baseline,entries_residual,entries_residual_std,entries_residual_z,entries_route_mean,entries_vs_route_mean,entries_z_score
0,2024-01-02,ADB_FCO,SMIRNE_ROMA,TUR_ITA,1,1,1,1,0,1,...,0.142857,0.856858,0.0,2.710084,-0.333333,2.180908,-0.313796,2.555556,0.846154,-0.313796
1,2024-02-19,ADD_FCO,ADDIS ABABA_ROMA,ETH_ITA,1,1,0,1,0,1,...,0.142857,0.856858,0.0,2.710084,-0.333333,2.180908,-0.313796,2.555556,0.846154,-0.313796
2,2024-02-26,AKL_FCO,AUCKLAND_ROMA,NZL_ITA,1,1,0,1,0,1,...,0.142857,0.856858,0.0,2.710084,-0.333333,2.180908,-0.313796,2.555556,0.846154,-0.313796
3,2024-01-01,AMM_FCO,AMMAN_ROMA,JOR_ITA,2,0,0,1,0,1,...,0.142857,0.856858,0.0,2.710084,-0.333333,2.180908,-0.313796,2.555556,0.846154,-0.313796
4,2024-05-01,AMM_FCO,AMMAN_ROMA,JOR_ITA,1,0,0,1,0,1,...,0.142857,0.856858,0.0,2.710084,-0.333333,2.180908,-0.313796,2.555556,0.846154,-0.313796


Required columns check: OK

[2] NUMERIC SANITY CHECKS
Non-negative count checks: OK

[3] DATA-QUALITY WARNINGS
Rows with data-quality issues: 0

[4] JSON SERIALIZATION TEST
Serialization/deserialization check: OK


### 7.3 Output

This cell saves the report as a markdown file.  
###### *In order to save the related table, uncomment the section*

In [35]:

AGENT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

airport = final_state["perimeter"].upper().strip()

report_path = AGENT_REPORT_DIR / f"{airport}_transit_anomaly_report.md"
report_path.write_text(final_state["final_report"], encoding="utf-8")

print(f"Report saved to: {report_path}")

# Save the scored route-day dataset and comparison-ready CSV.
# These files make the Multi-Agent output directly comparable with the airport-specific Classical output.
if "scored_dataframe_json" in final_state:
    scored_df = pd.read_json(
        io.StringIO(final_state["scored_dataframe_json"]),
        orient="records"
    )

    if not scored_df.empty:
        scored_df["date"] = pd.to_datetime(scored_df["date"], errors="coerce")
        scored_df["arrival_airport"] = airport

        comparison_cols = [
            "date",
            "arrival_airport",
            "route_airport",
            "route_city",
            "entries",
            "investigations",
            "flagged",
            "case_records",
            "flag_rate",
            "alarm_density_per_entry",
            "entries_dev_ratio7",
            "entries_residual_z",
            "iso_anomaly",
            "lof_anomaly",
            "zscore_anomaly",
            "consensus_anomaly",
            "rule_any",
            "final_anomaly",
            "evidence_type",
            "risk_score",
            "risk_level",
            "review_priority",
        ]

        comparison_cols = [c for c in comparison_cols if c in scored_df.columns]

        scored_path = AGENT_REPORT_DIR / f"{airport}_agent_scored_route_day.csv"
        ranked_path = AGENT_REPORT_DIR / f"{airport}_agent_ranked_anomaly_report.csv"
        comparison_path = AGENT_REPORT_DIR / f"{airport}_agent_comparison_ready.csv"

        scored_df.to_csv(scored_path, index=False)

        (
            scored_df.loc[scored_df["final_anomaly"] == 1]
            .sort_values(["risk_score", "model_agreement", "rule_count"], ascending=False)
            [comparison_cols]
            .to_csv(ranked_path, index=False)
        )

        scored_df[comparison_cols].to_csv(comparison_path, index=False)

        print(f"Scored route-day dataset saved to: {scored_path}")
        print(f"Ranked anomaly report saved to: {ranked_path}")
        print(f"Comparison-ready dataset saved to: {comparison_path}")
    else:
        print("No scored rows were available to export.")


Report saved to: c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\agent_report\FCO_transit_anomaly_report.md
Scored route-day dataset saved to: c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\agent_report\FCO_agent_scored_route_day.csv
Ranked anomaly report saved to: c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\agent_report\FCO_agent_ranked_anomaly_report.csv
Comparison-ready dataset saved to: c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\agent_report\FCO_agent_comparison_ready.csv


# Extra. Compare Classical and Multi-Agent Outputs on the Same Airport

This section compares the two pipelines fairly.

Instead of comparing the global Classical report with the airport-specific Multi-Agent report, we compare:

- Classical airport-specific output, for example `classical_FCO_comparison_ready.csv`;
- Multi-Agent airport-specific output, for example `FCO_agent_comparison_ready.csv`.

Since we do not have ground-truth anomaly labels, this is not an accuracy comparison.  
It is a consistency and operational-usefulness comparison.


In [36]:

airport = final_state["perimeter"].upper().strip()

classical_comparison_path = (
    IO_DIR
    / "classical_report"
    / f"classical_{airport}"
    / f"classical_{airport}_comparison_ready.csv"
)

agent_comparison_path = AGENT_REPORT_DIR / f"{airport}_agent_comparison_ready.csv"

if not classical_comparison_path.exists():
    print(f"Classical comparison file not found: {classical_comparison_path}")
    print("Run the airport-specific Classical report cell first.")
elif not agent_comparison_path.exists():
    print(f"Agent comparison file not found: {agent_comparison_path}")
    print("Run the Multi-Agent export cell first.")
else:
    classical_cmp = pd.read_csv(classical_comparison_path)
    agent_cmp = pd.read_csv(agent_comparison_path)

    classical_cmp["date"] = pd.to_datetime(classical_cmp["date"]).dt.strftime("%Y-%m-%d")
    agent_cmp["date"] = pd.to_datetime(agent_cmp["date"]).dt.strftime("%Y-%m-%d")

    key_cols = ["date", "route_airport"]

    classical_anom = classical_cmp[classical_cmp["final_anomaly"] == 1].copy()
    agent_anom = agent_cmp[agent_cmp["final_anomaly"] == 1].copy()

    classical_keys = set(map(tuple, classical_anom[key_cols].values))
    agent_keys = set(map(tuple, agent_anom[key_cols].values))

    overlap_keys = classical_keys & agent_keys

    comparison_summary = pd.DataFrame({
        "metric": [
            "airport",
            "classical_rows",
            "agent_rows",
            "classical_anomalies",
            "agent_anomalies",
            "overlapping_anomalies",
            "overlap_rate_vs_classical",
            "overlap_rate_vs_agent",
        ],
        "value": [
            airport,
            len(classical_cmp),
            len(agent_cmp),
            len(classical_anom),
            len(agent_anom),
            len(overlap_keys),
            round(len(overlap_keys) / max(len(classical_anom), 1), 4),
            round(len(overlap_keys) / max(len(agent_anom), 1), 4),
        ]
    })

    print("Comparison summary:")
    display(comparison_summary)

    overlap_df = agent_anom[
        agent_anom[key_cols].apply(tuple, axis=1).isin(overlap_keys)
    ].sort_values("risk_score", ascending=False)

    print("Overlapping anomalies:")
    display(overlap_df.head(20))

    comparison_report_path = AGENT_REPORT_DIR / f"{airport}_classical_vs_agent_comparison.md"

    comparison_report = f"""# Classical vs Multi-Agent Comparison — {airport}

## Comparison Scope

Both pipelines are compared on the same arrival airport: **{airport}**.

The comparison key is:

- `date`
- `route_airport`

This means an anomaly overlaps when both pipelines flag the same route on the same date.

## Summary

{comparison_summary.to_markdown(index=False)}

## Interpretation

This is not an accuracy comparison because there are no ground-truth anomaly labels.

Instead, this comparison checks:
- whether both pipelines flag a similar number of anomalies;
- whether they agree on the same route-day records;
- whether their ranking and risk-priority logic are operationally similar;
- whether the Multi-Agent report adds useful explanation without changing the underlying evidence.

## Overlapping Anomalies

{overlap_df.head(20).to_markdown(index=False) if not overlap_df.empty else "No overlapping anomalies found."}

## Suggested Conclusion

The Classical pipeline is stronger for reproducibility and static reporting.  
The Multi-Agent pipeline is stronger for interactivity and user-specific explanations.  
The best operational setup is to use the Classical pipeline as the stable analytical baseline and the Multi-Agent pipeline as an interactive explanation layer.
"""

    comparison_report_path.write_text(comparison_report, encoding="utf-8")

    print(f"Comparison report saved to: {comparison_report_path}")


Comparison summary:


,metric,value
0,airport,FCO
1,classical_rows,579
2,agent_rows,579
3,classical_anomalies,28
4,agent_anomalies,130
5,overlapping_anomalies,25
6,overlap_rate_vs_classical,0.8929
7,overlap_rate_vs_agent,0.1923


Overlapping anomalies:


,date,arrival_airport,route_airport,route_city,entries,investigations,flagged,case_records,flag_rate,alarm_density_per_entry,...,iso_anomaly,lof_anomaly,zscore_anomaly,consensus_anomaly,rule_any,final_anomaly,evidence_type,risk_score,risk_level,review_priority
391,2024-02-28,FCO,JED_FCO,JEDDAH_ROMA,64,0,0,0,0.000000,0.000000,...,1,1,1,1,1,1,Model + rules,75.00,CRITICAL,P1 - immediate review
354,2024-02-24,FCO,SAW_FCO,ISTANBUL_ROMA,12,12,1,0,0.083333,0.000000,...,1,0,1,1,1,1,Model + rules,63.33,CRITICAL,P1 - immediate review
83,2024-01-18,FCO,IKA_FCO,TEHRAN_ROMA,21,21,1,0,0.047619,0.000000,...,0,1,1,1,1,1,Model + rules,44.55,HIGH,P1 - immediate review
153,2024-01-25,FCO,IKA_FCO,TEHRAN_ROMA,39,39,0,0,0.000000,0.000000,...,0,1,1,1,1,1,Model + rules,44.55,HIGH,P1 - immediate review
237,2024-02-12,FCO,TIA_FCO,TIRANA_ROMA,23,0,0,0,0.000000,0.000000,...,1,0,1,1,1,1,Model + rules,43.43,HIGH,P1 - immediate review
171,2024-01-26,FCO,TIA_FCO,TIRANA_ROMA,353,353,16,0,0.045326,0.000000,...,1,0,1,1,0,1,Model consensus only,42.55,HIGH,P1 - immediate review
309,2024-02-19,FCO,TIA_FCO,TIRANA_ROMA,44,44,3,0,0.068182,0.000000,...,1,0,1,1,0,1,Model consensus only,38.04,MODERATE,P2 - high priority
179,2024-01-27,FCO,LGW_FCO,LONDRA_ROMA,4,3,0,0,0.000000,0.000000,...,0,1,0,0,1,1,Business rules only,37.70,MODERATE,P2 - high priority
367,2024-02-25,FCO,TIA_FCO,TIRANA_ROMA,75,25,6,0,0.080000,0.000000,...,1,0,1,1,0,1,Model consensus only,35.53,MODERATE,P2 - high priority
374,2024-02-26,FCO,MLE_FCO,MALE_ROMA,8,8,2,1,0.250000,0.125000,...,0,1,0,0,1,1,Business rules only,32.88,MODERATE,P2 - high priority


Comparison report saved to: c:\Users\AFGWO\OneDrive\Desktop\Lectures\Machine Learning\reply-classical-vs-multiagent\src\io\agent_report\FCO_classical_vs_agent_comparison.md
